# 00b · 몬테카를로 오차와 신뢰구간 (Monte Carlo Error and Confidence Intervals)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Ross 12e §2.7 (체비쇼프·LLN·CLT) · §11.1 (시뮬레이션 입문) · §11.6.1 (안티테틱 변량) · §11.7 (필요한 실행 횟수) · G&S 4e §5.10 (LLN·CLT), §7.4 (큰 수의 법칙) · 보조: Owen, *Monte Carlo theory, methods and examples* Ch. 2 (오차 추정·신뢰구간), Ch. 8 (분산 감소) |
| 선수 노트북 | 00a |
| 예상 소요 | 75분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **시뮬레이션 결과는 "숫자"가 아니라 "숫자 ± 표준오차"다.** 오차는 $1/\sqrt n$으로 줄고, 참값과의 거리를 SE 단위(z)로 읽어야 "맞았다/틀렸다"를 말할 수 있다. 이 과정의 모든 노트북이 쓰는 `mc_estimate`, `mc_proportion`, `compare_table`, `assert_close(k=4)`의 근거를 여기서 만든다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm, binom
from scipy.special import comb
from scipy.integrate import quad
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.plots import plot_hist_vs_pdf

SEED, rng = rng_for("00b")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (00a) 단순 랜덤 워크 $S_n$의 분포는? 왜 정수 폭 1 칸 히스토그램이 pmf의 절반 높이로 보였나?

<details><summary>정답</summary>

$(S_n+n)/2\sim\mathrm{Bin}(n,p)$. $S_n$은 $n$과 홀짝이 같아 칸의 절반이 비고, `density=True` 정규화가 채워진 칸을 절반 높이로 만든다.

</details>

**Q2.** (00a) $N=20$, $i=10$에서 $p=1/2$와 $p=0.45$의 승리 확률은 각각? 두 값이 왜 그렇게 다른가?

<details><summary>정답</summary>

$0.5$와 $0.1185$. $h(i)=\dfrac{1-(q/p)^i}{1-(q/p)^N}$에서 $(q/p)^i$가 지수적으로 커지기 때문이다. 5% 불리함이 10걸음·20걸음에 걸쳐 복리로 쌓인다.

</details>

**Q3.** (00a) 파산 시뮬레이션을 고정된 걸음 수만 돌리면 무엇이 잘못되나?

<details><summary>정답</summary>

아직 흡수되지 않은 경로를 임의로 분류하게 되어 추정이 **편향**된다. 모든 경로가 흡수될 때까지 돌리고 `active.size == 0`을 assert했다. 오늘의 주제인 표준오차는 편향이 없을 때만 오차의 전부다.

</details>

**Q4 (연결 고리).** (학부 확률) i.i.d. 표본평균 $\bar X_n$의 분산이 $\sigma^2/n$인 이유는? 체비쇼프 부등식은?

<details><summary>정답</summary>

독립이면 합의 분산은 분산의 합 $n\sigma^2$이고, $1/n$을 곱한 것의 분산이므로 $n^2$으로 나눈다. 체비쇼프: $P(|X-EX|\ge\varepsilon)\le\mathrm{Var}(X)/\varepsilon^2$. — 이 두 줄이 오늘 정리 A의 전부다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **몬테카를로 추정량(Monte Carlo estimator).** $\theta=E[f(X)]$를 i.i.d. 표본 $X_1,\dots,X_n$으로
  $$\hat\theta_n=\frac1n\sum_{i=1}^n f(X_i)$$
  로 추정한다. 이 노트북 내내 $\sigma^2=\mathrm{Var}(f(X))<\infty$를 가정한다.
- **표준오차(standard error, SE).** $\mathrm{SE}=\sigma/\sqrt n$. $\sigma$를 모르므로 추정치
  $$\widehat{\mathrm{SE}}=\frac{s}{\sqrt n},\qquad s^2=\frac1{n-1}\sum_{i=1}^n\bigl(f(X_i)-\hat\theta_n\bigr)^2\ (\texttt{ddof=1})$$
  를 쓴다. `spkit.mc.mc_estimate`가 정확히 이것을 계산한다.
- **베르누이(비율) 특수형.** $f\in\{0,1\}$이면 $\sigma^2=p(1-p)$, $\widehat{\mathrm{SE}}=\sqrt{\hat p(1-\hat p)/n}$ (`spkit.mc.mc_proportion`). $\hat p\in\{0,1\}$이면 이 식이 0이 되므로 $1/n$을 하한으로 둔다. 최악의 경우는 $p=1/2$에서 $\sigma^2=1/4$.
- **95% 신뢰구간(Wald 구간).** $\hat\theta_n\pm1.96\,\widehat{\mathrm{SE}}$ (`MCResult.ci95`). **포함률(coverage)**: 같은 실험을 $R$번 반복했을 때 구간이 참값 $\theta$를 덮는 비율. 명목값(nominal)은 0.95이지만 실제값은 $n$과 분포에 따라 다르다.
- **RMSE(root mean squared error).** $\sqrt{E[(\hat\theta_n-\theta)^2]}$. 불편 추정량이므로 $=\sqrt{\mathrm{Var}(\hat\theta_n)}=\sigma/\sqrt n$. 로그-로그에서 $\log\mathrm{RMSE}=\log\sigma-\tfrac12\log n$: **기울기 $-1/2$**.
- **필요 표본 수.** 반폭 $\varepsilon$의 95% 구간을 원하면 $1.96\sigma/\sqrt n=\varepsilon$, 즉 $n\approx(1.96\sigma/\varepsilon)^2$. 정밀도 10배 = 표본 100배 (Ross 12e §11.7).
- **안티테틱 변량(antithetic variates).** $Z$와 $-Z$(또는 $U$와 $1-U$)를 짝지어 $A=\tfrac12[f(Z)+f(-Z)]$의 평균을 쓰는 추정량. $f$가 단조이면 두 항의 공분산이 음수라 분산이 준다 (Ross 12e §11.6.1).

### 2.2 정리 A — 불편성, 분산, 체비쇼프 → 약한 큰 수의 법칙 (Ross 12e §2.7, G&S 4e §7.4)

> $X_i$ i.i.d., $\sigma^2=\mathrm{Var}(f(X))<\infty$이면
> $$E[\hat\theta_n]=\theta,\qquad \mathrm{Var}(\hat\theta_n)=\frac{\sigma^2}n,\qquad P(|\hat\theta_n-\theta|\ge\varepsilon)\le\frac{\sigma^2}{n\varepsilon^2}\xrightarrow[n\to\infty]{}0\quad(\forall\varepsilon>0).$$

**가정이 왜 필요한가**
- **독립성.** $\mathrm{Var}\bigl(\sum f(X_i)\bigr)=\sum_i\mathrm{Var}+\sum_{i\ne j}\mathrm{Cov}$에서 공분산 항을 0으로 만든다. 상관된 표본(E3의 AR(1), 나중의 MCMC)에서는 $\sigma^2/n$이 **과소평가**가 되어 구간이 너무 좁아진다.
- **동일분포.** 평균이 모두 $\theta$여야 $E[\hat\theta_n]=\theta$ (불편성).
- **유한 분산 $\sigma^2<\infty$.** 체비쇼프의 우변이 정의되려면 필요하다. 무한 분산이지만 평균은 유한한 경우 LLN 자체는 여전히 성립하지만(G&S 4e §7.4), 수렴 속도가 $n^{-1/2}$보다 느리다 (E3 선택 문제의 Pareto).

<details><summary>증명 (완전 증명, 세 줄)</summary>

1. **불편성.** 기댓값의 선형성: $E[\hat\theta_n]=\frac1n\sum_i E[f(X_i)]=\frac1n\cdot n\theta=\theta$.
2. **분산.** 독립이므로 $\mathrm{Var}\bigl(\sum_i f(X_i)\bigr)=\sum_i\mathrm{Var}(f(X_i))=n\sigma^2$. 상수 $1/n$을 빼내면 제곱이 되어 $\mathrm{Var}(\hat\theta_n)=n\sigma^2/n^2=\sigma^2/n$.
3. **체비쇼프.** $Y=\hat\theta_n-\theta$라 두자. 모든 결과에서 $\varepsilon^2\,\mathbf 1\{|Y|\ge\varepsilon\}\le Y^2$이다 ($|Y|\ge\varepsilon$이면 좌변 $\varepsilon^2\le Y^2$, 아니면 좌변 0). 양변에 기댓값을 취하면 $\varepsilon^2P(|Y|\ge\varepsilon)\le E[Y^2]=\mathrm{Var}(\hat\theta_n)=\sigma^2/n$. $\varepsilon^2$으로 나누면 끝. $\square$

</details>

### 2.3 정리 B — CLT와 신뢰구간의 점근 타당성 (G&S 4e §5.10, Ross 12e §2.7)

> $\dfrac{\sqrt n(\hat\theta_n-\theta)}{\sigma}\Rightarrow N(0,1)$이고 $s\to\sigma$ a.s.이므로
> $$P\bigl(\theta\in\hat\theta_n\pm1.96\,s/\sqrt n\bigr)\longrightarrow0.95.$$

**가정이 왜 필요한가**
- **i.i.d.와 유한 분산.** CLT의 가정. 유한 분산이 없으면 정규화된 합은 정규가 아닌 안정분포(stable law)로 가고, Wald 구간의 포함률은 0.95로 수렴하지 않는다.
- **$n$이 "충분히" 커야 한다.** 유한 $n$에서는 근사다. 베르누이 $p=0.05$, $n=100$이면 $np=5$뿐이라 정규 근사가 나쁘고 실제 포함률은 약 0.877이다(정리 C로 정확히 계산 가능).

**증명 (인용).** CLT는 특성함수로 증명한다 (G&S 4e §5.10). $s^2\to\sigma^2$ a.s.는 강한 큰 수의 법칙을 $f(X)^2$과 $f(X)$에 적용한 뒤 $s^2=\frac{n}{n-1}\bigl(\overline{f^2}-\bar f^2\bigr)$로 결합. 두 수렴은 Slutsky 정리로 합쳐진다. 이 노트북에서는 인용하고 4.3절의 포함률 실험으로 확인한다.

### 2.4 정리 C — Wald 구간의 정확한 포함률 (이항 경우)

> $K\sim\mathrm{Bin}(n,p)$, $\hat p=K/n$, $\widehat{\mathrm{SE}}(k)=\max\bigl(\sqrt{\hat p(1-\hat p)/n},\,1/n\bigr)$일 때 포함률은 유한합
> $$C(p,n)=\sum_{k=0}^n\binom nk p^k(1-p)^{n-k}\;\mathbf 1\Bigl\{\bigl|\tfrac kn-p\bigr|\le1.96\,\widehat{\mathrm{SE}}(k)\Bigr\}.$$
> 예: $C(0.3,100)\approx0.9502$, $C(0.05,100)\approx0.8775$ (4.3절에서 코드로 계산).

**가정이 왜 필요한가**
- **구간 규칙이 $k$만의 함수.** 그래서 포함 사건이 $K$의 값으로 결정되고 유한합으로 **정확히** 계산된다. `mc_proportion`의 $1/n$ 보정을 그대로 넣어야 시뮬레이션과 "같은 규칙"을 비교하는 것이 된다.

<details><summary>증명 (완전 증명, 두 줄)</summary>

포함 사건 $\{\theta\in\text{구간}\}=\{|K/n-p|\le1.96\,\widehat{\mathrm{SE}}(K)\}$은 $K$의 값만으로 결정되는 사건이므로, 그 확률은 $K$의 pmf로 가중한 지시함수의 합이다: $P(\text{포함})=\sum_k P(K=k)\,\mathbf 1\{k\text{가 조건을 만족}\}$. $P(K=k)=\binom nk p^k(1-p)^{n-k}$를 넣으면 식이 된다. $\square$

두 줄짜리 증명이지만 교훈은 크다: "닫힌 형식"이 없는 양도 **유한 열거**로 정확히 계산할 수 있고, 그러면 MC 결과를 4 SE 검사에 걸 수 있다.

</details>

### 2.5 정리 D — 안티테틱 짝의 분산 (Ross 12e §11.6.1)

> $A=\tfrac12[f(Z)+f(-Z)]$, $\sigma^2=\mathrm{Var}(f(Z))$이면
> $$\mathrm{Var}(A)=\tfrac12\bigl[\sigma^2+\mathrm{Cov}(f(Z),f(-Z))\bigr].$$
> $f(z)=z^+=\max(z,0)$, $Z\sim N(0,1)$이면 $\mathrm{Cov}=-\dfrac1{2\pi}$, $\mathrm{Var}(A)=\tfrac14\bigl(1-\tfrac2\pi\bigr)\approx0.0908$, 같은 정규 난수 개수 기준 분산비 $2\mathrm{Var}(A)/\sigma^2\approx0.533$.

**가정이 왜 필요한가**
- **$f(Z)$와 $f(-Z)$의 공분산이 음수여야 이득이다.** $f$가 단조이면(또는 여기처럼 $Z^+Z^-=0$이면) 보장된다. 짝의 두 항이 양의 상관이면 오히려 손해다 — 예: $f(z)=z^2$이면 $f(Z)=f(-Z)$라 분산이 전혀 줄지 않는다(짝이 같은 난수 두 개를 세는 셈).

<details><summary>증명 (완전 증명; 숫자 부분은 E2에서 학습자가 완성)</summary>

$U=f(Z)$, $V=f(-Z)$는 같은 분포($-Z\overset d=Z$)라 $\mathrm{Var}U=\mathrm{Var}V=\sigma^2$. 일반 공식 $\mathrm{Var}\bigl(\tfrac12(U+V)\bigr)=\tfrac14(\mathrm{Var}U+\mathrm{Var}V+2\mathrm{Cov}(U,V))=\tfrac12[\sigma^2+\mathrm{Cov}(U,V)]$.

$f(z)=z^+$일 때: $Z^+\cdot(-Z)^+=Z^+Z^-=0$ (둘 중 하나는 항상 0)이므로 $E[UV]=0$이고 $\mathrm{Cov}=0-E[Z^+]E[Z^-]=-(1/\sqrt{2\pi})^2=-1/(2\pi)$. $E[(Z^+)^2]=\tfrac12E[Z^2]=\tfrac12$이므로 $\sigma^2=\tfrac12-\tfrac1{2\pi}$. 대입하면 $\mathrm{Var}(A)=\tfrac12[\tfrac12-\tfrac1{2\pi}-\tfrac1{2\pi}]=\tfrac14-\tfrac1{2\pi}=\tfrac14(1-2/\pi)$. 짝 하나가 정규 난수 **두 개**를 쓰므로 공정한 비교는 $\mathrm{Var}(A)/(\sigma^2/2)=2\mathrm{Var}(A)/\sigma^2$. $\square$

</details>

### 2.6 직관

MC 오차는 표본 크기의 **제곱근**에 반비례한다. 따라서 그림에서 "대충 맞아 보임"은 근거가 아니고, 오차막대(1.96 SE)와 참값의 거리를 SE 단위로 읽어야 한다. 3 SE 밖이면 버그이거나 가정 오해다(운일 확률 0.3%). `assert_close(k=4)`는 표본이 줄어(FAST) SE가 커지면 허용 폭도 같이 커지므로 어느 크기에서나 거짓 경보 확률이 약 $6\times10^{-5}$($=P(|N(0,1)|>4)$)로 같다.

신뢰구간 95%는 "이 구간에 참값이 있을 확률"이 아니라 "이 **절차**를 반복하면 95%의 구간이 참값을 덮는다"는 빈도적 진술이며, 그 95%조차 근사다.

MC가 닫힌 형식을 이기는 순간은 적분이 고차원이거나 사건이 경로 전체에 걸쳐 정의될 때다: 코드 두 줄이 수치적분·조합론을 대신한다(4.5절).

### 2.7 함정(traps)

1. `np.std`의 기본은 `ddof=0`이다. SE에는 `ddof=1`이 관례(`mc_estimate`가 그렇게 한다). $n$이 크면 차이는 미미하지만 코드 리뷰 포인트.
2. "오차가 $1/n$으로 준다"는 착각: $1/\sqrt n$이다. 표본 10배 = 오차 $\sqrt{10}\approx3.16$배 감소.
3. 베르누이 $p$가 0이나 1 근처일 때 $\hat p=0$이면 $\widehat{\mathrm{SE}}=0$: 구간 폭이 0이 되어 참값을 절대 못 덮는다. `mc_proportion`은 $1/n$ 하한을 둔다.
4. 포함률을 하나의 실험으로 판단하려는 시도: 포함률은 **반복 실험의 성질**이다. 한 구간이 참값을 덮었는지는 0 또는 1이다.
5. 상관된 표본(같은 경로에서 연속 시각의 값, MCMC)에 $s/\sqrt n$을 쓰면 SE가 크게 과소평가된다 (E3: $\rho=0.9$ AR(1)에서 약 4.3배).
6. 정규 난수 $M\times N$ 배열을 모든 $N$에 대해 한 번에 만들면 메모리가 터진다. $N$ 격자별로 따로 생성하고 즉시 축약한다(4.2절).

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

- 정리 A의 세 줄 증명을 안 보고 다시 써 보세요. 어느 줄에서 독립성이 쓰이나요?
- "95% 신뢰구간"을 빈도적 진술로 한 문장으로 적어 보세요.

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향(0.95보다 위/아래 등)이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요.

In [ ]:
predictions = {
    # 단위 정사각형 안의 균등점이 단위원 사분면에 들어갈 확률 pi/4 를 n=1e5 개로 추정할 때 SE 는?
    # (0.1? 0.01? 0.001?)
    "se_pi_quarter": None,
    # RMSE vs N 로그-로그 그림의 기울기는?
    "loglog_slope": None,
    # p=0.3, n=100 베르누이 표본의 Wald 95% 구간이 참값을 덮는 실제 비율은? (0.95 보다 클까 작을까?)
    "coverage_p03_n100": None,
    # p=0.05, n=100 이면 포함률은?
    "coverage_p005_n100": None,
    # 표준정규 10 개의 최댓값의 기댓값 E[max(Z_1..Z_10)] 은? (1? 1.5? 2? 3?)
    "emax10": None,
    # E[Z+] 추정에서 안티테틱 짝 (Z, -Z) 를 쓰면 같은 정규 난수 개수 기준으로 분산이 몇 배가 되나?
    # (0.9? 0.5? 0.1?)
    "var_ratio_antithetic": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 첫 추정과 오차막대

(a) 베르누이: 단위 정사각형의 균등점이 사분원 안에 들어갈 확률 $\pi/4$. (b) 연속: $E[Z^+]=1/\sqrt{2\pi}$. 두 경우 모두 "추정 ± 1.96 SE"를 출력하고, 참값과의 거리를 SE 단위 $z=(\hat\theta-\theta)/\widehat{\mathrm{SE}}$로 읽는다. 보통 $|z|<2$여야 한다.

In [ ]:
n1 = scale(100_000)                       # FAST: 10 000
U = rng.random((n1, 2))
inside = (U**2).sum(axis=1) <= 1.0

fig, ax = plt.subplots(figsize=(4.2, 4.2))
show = slice(0, 2000)
ax.scatter(*U[show][inside[show]].T, s=3, alpha=0.5, label="inside")
ax.scatter(*U[show][~inside[show]].T, s=3, alpha=0.5, label="outside")
tt = np.linspace(0, np.pi / 2, 200)
ax.plot(np.cos(tt), np.sin(tt), "k", lw=1)
ax.set_aspect("equal"); ax.set_title("First 2000 of the uniform points"); ax.legend(loc="lower left", markerscale=4)
plt.show()

In [ ]:
est_pi = mc_proportion(inside)
theta_pi = np.pi / 4
z_pi = (est_pi.mean - theta_pi) / est_pi.se
print(f"(a) pi/4 : {est_pi}   ci95={tuple(round(c, 4) for c in est_pi.ci95)}")
print(f"    exact {theta_pi:.6f},  z = {z_pi:+.2f} SE")

Z = rng.standard_normal(n1)
est_zp = mc_estimate(np.maximum(Z, 0))
theta_zp = 1 / np.sqrt(2 * np.pi)
z_zp = (est_zp.mean - theta_zp) / est_zp.se
print(f"(b) E[Z+]: {est_zp}   ci95={tuple(round(c, 4) for c in est_zp.ci95)}")
print(f"    exact {theta_zp:.6f},  z = {z_zp:+.2f} SE")

`MCResult`의 문자열이 곧 이 과정의 보고 형식이다: `평균 ± SE (n=...)`. 두 $z$ 값이 $[-2,2]$ 안이면 정상, 3 밖이면 (0.3%의 운이 아니라면) 버그다.

### 4.2 오차 vs $N$: 로그-로그 기울기 $-1/2$

$E[Z^+]$ 추정을 격자 $N\in\{10^2,10^{2.5},\dots\}$마다 $R$번 독립 반복해 RMSE를 측정한다. 함정 6: $N$마다 따로 생성하고 즉시 평균으로 축약한다(가장 큰 배열은 $R\times N_{\max}$). 기울기의 SE는 유도된 근사식을 쓴다: $\mathrm{RMSE}_N^2\approx(\sigma^2/N)\chi^2_R/R$이므로 $\mathrm{sd}(\log\mathrm{RMSE}_N)\approx\tfrac12\sqrt{2/R}=1/\sqrt{2R}$, 회귀 기울기의 SE는 $(1/\sqrt{2R})\big/\sqrt{\sum_j(x_j-\bar x)^2}$, $x_j=\log N_j$.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def rmse_vs_n(Ns, R, rng, theta, f=lambda z: np.maximum(z, 0)):
    """N 마다 R 개의 독립 MC 추정을 만들어 RMSE 를 돌려준다 (큰 배열을 한 번에 만들지 않는다)."""
    out = []
    for N in Ns:
        est = f(rng.standard_normal((R, int(N)))).mean(axis=1)
        out.append(np.sqrt(np.mean((est - theta) ** 2)))
    return np.array(out)

def slope_se(Ns, R):
    """log-log 회귀 기울기의 표준오차 (RMSE^2 ~ (sigma^2/N) chi2_R / R 근사)."""
    x = np.log(Ns)
    return (1 / np.sqrt(2 * R)) / np.sqrt(((x - x.mean()) ** 2).sum())

def wald_coverage_exact(p, n, z=1.96):
    """Wald 95% 구간의 정확한 포함률 (정리 C; mc_proportion 의 1/n 보정과 같은 규칙)."""
    k = np.arange(n + 1)
    phat = k / n
    se = np.maximum(np.sqrt(phat * (1 - phat) / n), 1 / n)
    return float((binom.pmf(k, n, p) * (np.abs(phat - p) <= z * se)).sum())

In [ ]:
Ns_all = (10 ** np.arange(2, 5.01, 0.5)).astype(int)      # 100 ... 100000 (7점)
Ns = Ns_all                                                # FAST 에서도 같은 격자(R만 1/4)
R = scale(200, divisor=4, minimum=50)                      # FAST: 50 (총 난수 FULL 2.9e7 / FAST 7.3e6)
sigma_zp = np.sqrt(0.5 - 1 / (2 * np.pi))                  # sd(Z+) = 0.5838

rmse = rmse_vs_n(Ns, R, rng, theta_zp)
slope, intercept = np.polyfit(np.log(Ns), np.log(rmse), 1)
se_slope = slope_se(Ns, R)
print(f"Ns={Ns.tolist()}, R={R}")
print(f"fitted slope = {slope:+.4f} ± {se_slope:.4f}   (theory -0.5, z = {(slope + 0.5) / se_slope:+.2f})")

fig, ax = plt.subplots()
ax.loglog(Ns, rmse, "o", label=f"MC RMSE (R={R} replicates per N)")
ax.loglog(Ns, sigma_zp / np.sqrt(Ns), "-", label=r"theory $\sigma/\sqrt{N}$")
ax.loglog(Ns, np.exp(intercept) * Ns**slope, "--", label=f"fit: slope = {slope:+.3f}")
ax.set_xlabel("sample size N"); ax.set_ylabel("RMSE of the estimate of E[Z+]")
ax.set_title("Monte Carlo error vs sample size"); ax.legend()
plt.show()

축을 로그로 바꾸면 "$1/\sqrt N$"이 기울기 $-1/2$의 직선이 된다. 점이 이론선 위에 얹히는 것은 추정량이 불편이라 RMSE $=$ SE라는 뜻이다(편향이 있으면 점들이 직선에서 떠서 $N\to\infty$에서 평평해진다). $N$이 $100$배 늘 때마다 RMSE는 $10$배 준다(격자 $10^2\to10^4$에서 $1/10$, $10^2\to10^5$에서 $1/\sqrt{1000}\approx1/31.6$): **자릿수 하나 = 표본 100배**.

### 4.3 신뢰구간의 포함률

(a) 그림: $p=0.3$, $n=100$ 실험을 100번 반복해 각 구간 `mc_proportion(...).ci95`를 세로선으로 그린다. 참값을 못 덮는 구간은 빨간색이다. 기대 개수는 $100\times0.05=5$개이지만 개수 자체가 $\mathrm{Bin}(100,0.05)$ 확률변수라 3개나 9개도 흔하다 — 100번으로는 포함률을 두 자리도 못 읽는다(함정 4). 그래서 (b)에서 $R$을 훨씬 크게 잡는다.

In [ ]:
p_show, n_show = 0.3, 100
fig, ax = plt.subplots(figsize=(9, 3.6))
n_miss = 0
for i in range(100):
    r = mc_proportion(rng.random(n_show) < p_show)
    lo, hi = r.ci95
    miss = not (lo <= p_show <= hi)
    n_miss += miss
    ax.plot([i, i], [lo, hi], color="C3" if miss else "C0", lw=1.5, alpha=0.9)
    ax.plot(i, r.mean, ".", color="C3" if miss else "C0", ms=4)
ax.axhline(p_show, color="k", lw=1, ls="--", label=f"true p = {p_show}")
ax.set_xlabel("replicate"); ax.set_ylabel("Wald 95% CI for p")
ax.set_title(f"100 Wald 95% CIs, p = {p_show}, n = {n_show}: {n_miss} miss the true p (red)")
ax.legend(loc="upper right")
plt.show()

(b) 정량: 실험을 $R$번 반복하는 것은 $K\sim\mathrm{Bin}(n,p)$를 $R$개 뽑는 것과 같다. `mc_proportion`과 같은 규칙(1/n 하한)으로 구간을 만들고 포함률을 추정한 뒤, 정리 C의 정확값과 비교한다.

In [ ]:
def wald_coverage_mc(p, n, R, rng, z=1.96):
    """Bin(n,p) 를 R 번 뽑아 Wald 구간(1/n 하한)의 포함 여부를 돌려준다 (bool 배열)."""
    K = rng.binomial(n, p, size=R)
    phat = K / n
    se = np.maximum(np.sqrt(phat * (1 - phat) / n), 1 / n)
    return np.abs(phat - p) <= z * se

R_cov, n_cov = scale(20_000), 100                          # FAST: R = 2000
cov_p03 = mc_proportion(wald_coverage_mc(0.3, n_cov, R_cov, rng))
cov_p005 = mc_proportion(wald_coverage_mc(0.05, n_cov, R_cov, rng))
exact_p03 = wald_coverage_exact(0.3, n_cov)
exact_p005 = wald_coverage_exact(0.05, n_cov)
print(f"p=0.30, n=100: MC coverage {cov_p03}   exact {exact_p03:.5f}")
print(f"p=0.05, n=100: MC coverage {cov_p005}   exact {exact_p005:.5f}")

$p=0.3$에서는 명목값 0.95에 거의 정확히 맞지만, $p=0.05$에서는 0.88 근처다: $np=5$뿐이라 정규 근사가 나쁘고, $K$가 작을 때 $\sqrt{\hat p(1-\hat p)/n}$ 자체가 과소평가된다. 포함률은 하나의 실험에서 볼 수 없고(함정 4), $R$번 반복해야 비로소 "비율"이 된다.

(c) $n$에 따른 포함률 곡선. 선은 열거로 계산한 정확값, 점은 MC(오차막대 1.96 SE). 0.95로 가긴 하지만 단조가 아니다 — 이항의 이산성 때문에 톱니 모양이 난다.

In [ ]:
n_grid = np.array([10, 20, 50, 100, 200, 500, 1000, 2000])
fig, ax = plt.subplots()
for p, c in ((0.05, "C3"), (0.3, "C0")):
    exact_curve = [wald_coverage_exact(p, n) for n in n_grid]
    mc_curve = [mc_proportion(wald_coverage_mc(p, n, R_cov, rng)) for n in n_grid]
    ax.plot(n_grid, exact_curve, "-", color=c, label=f"exact, p = {p}")
    ax.errorbar(n_grid, [m.mean for m in mc_curve], yerr=[1.96 * m.se for m in mc_curve],
                fmt="o", color=c, ms=4, capsize=2, label=f"MC (R={R_cov}), p = {p}")
ax.axhline(0.95, color="k", lw=1, ls="--", label="nominal 0.95")
ax.set_xscale("log"); ax.set_xlabel("n"); ax.set_ylabel("coverage of the Wald 95% CI")
ax.set_title("Actual coverage of the Wald interval vs n"); ax.legend(loc="lower right", fontsize=8)
plt.show()

### 4.4 `SPKIT_FAST` 스위치와 `assert_close(k=4)`

이 과정의 모든 노트북은 환경변수 `SPKIT_FAST`가 켜지면 표본을 `scale()`로 줄여서 실행된다(CI와 빠른 확인용). 값을 바꾸지는 말고 읽기만 한다.

In [ ]:
print(f"os.environ.get('SPKIT_FAST') = {os.environ.get('SPKIT_FAST')!r}  ->  fast() = {fast()}")
print(f"scale(20000) = {scale(20000)}")
print(f"scale(200)   = {scale(200)}      (minimum=200 이라 FAST 에서도 200)")
print(f"scale(20000, divisor=100) = {scale(20000, divisor=100)}")

FAST에서는 표본이 1/10이라 SE가 $\sqrt{10}\approx3.16$배 커진다. 그래도 `assert_close(k=4)`는 **SE에 상대적**이므로 허용 폭 $4\,\widehat{\mathrm{SE}}$도 같이 커지고, 거짓 경보 확률은 표본 크기와 무관하게 $P(|N(0,1)|>4)\approx6.3\times10^{-5}$로 같다. 즉 FAST 검사는 "덜 정밀하지만 똑같이 타당한" 검사다. 잡는 버그의 크기만 달라진다: FULL이 잡는 $4\,\mathrm{SE}$ 크기의 편향을 FAST는 놓칠 수 있다. 이 노트북의 실행 시간 목표: FAST < 15 s, FULL < 60 s.

### 4.5 닫힌 형식이 번거로울 때 MC가 이긴다

(a) $E[\max(Z_1,\dots,Z_{10})]$. 참고값은 순서통계량 밀도 $10\,\varphi(x)\Phi(x)^9$의 1차원 수치적분이 필요하지만 MC는 한 줄이다.

In [ ]:
n_max10 = scale(200_000)                                   # FAST: 20 000
M = rng.standard_normal((n_max10, 10)).max(axis=1)
est_max10 = mc_estimate(M)
emax10_ref = quad(lambda x: x * 10 * norm.pdf(x) * norm.cdf(x) ** 9, -np.inf, np.inf)[0]
print(f"E[max of 10 N(0,1)]: MC {est_max10}   quad {emax10_ref:.6f}   z = {(est_max10.mean - emax10_ref) / est_max10.se:+.2f}")

x = np.linspace(-1.5, 4.5, 400)
ax = plot_hist_vs_pdf(M, x, 10 * norm.pdf(x) * norm.cdf(x) ** 9, bins=60,
                      label_pdf=r"$10\,\varphi(x)\Phi(x)^9$")
ax.axvline(est_max10.mean, color="C2", ls="--", label=f"MC mean = {est_max10.mean:.3f}")
ax.set_xlabel("max of 10 standard normals"); ax.set_title("Maximum of 10 N(0,1) samples"); ax.legend()
plt.show()

(b) 정규 걸음 랜덤 워크 $S_k=\sum_{j\le k}\xi_j$가 20걸음 **내내** 양수일 확률. 사건이 경로 전체에 걸쳐 있어 조합론이 필요하지만 코드는 두 줄이다. 정확값은 Sparre Andersen 정리: 걸음이 대칭·연속이면 분포에 무관하게 $P(S_1>0,\dots,S_n>0)=\binom{2n}{n}/4^n$, 여기서는 $n=20$ 걸음이므로 $\binom{40}{20}/4^{20}\approx0.1254$.

In [ ]:
n_walk = scale(100_000)                                    # FAST: 10 000
S = rng.standard_normal((n_walk, 20)).cumsum(axis=1)
est_stay_pos = mc_proportion((S > 0).all(axis=1))
stay_pos_exact = comb(40, 20) / 4**20
print(f"P(S_k > 0, k=1..20): MC {est_stay_pos}   exact {stay_pos_exact:.6f}   z = {(est_stay_pos.mean - stay_pos_exact) / est_stay_pos.se:+.2f}")

이 값이 걸음 분포에 무관하다는 사실은 MC로 먼저 "발견"할 수 있다: 위 셀의 `rng.standard_normal`을 `rng.uniform(-1, 1, ...)`이나 `rng.standard_t(3, ...)`로 바꿔 보세요. 같은 0.1254가 나온다(정리는 05 모듈의 마팅게일·조합론에서 다시 만난다). 왜 $\pm1$ 걸음(00a)은 안 되는지도 생각해 보세요 — $S_k=0$이 양의 확률로 생기기 때문이다.

### 4.6 분산 감소 맛보기(teaser): 안티테틱 변량

$E[Z^+]$를 짝 $(Z,-Z)$로 추정하면 $A=\tfrac12[Z^++(-Z)^+]=|Z|/2$. 짝 하나가 정규 난수 두 개를 쓰므로 4.1(b)의 plain 추정(난수 $n_1$개)과 **같은 난수 개수** 기준으로 비교한다: 짝 $n_1/2$개.

In [ ]:
n_pairs = n1 // 2                                          # plain 과 같은 정규 난수 개수 (FAST: 5000 짝)
Zp = rng.standard_normal(n_pairs)
A = (np.maximum(Zp, 0) + np.maximum(-Zp, 0)) / 2           # = |Zp| / 2
est_anti = mc_estimate(A)
varA_exact = 0.25 * (1 - 2 / np.pi)
var_ratio_exact = 2 * varA_exact / sigma_zp**2
print(f"plain      : {est_zp}   (n={n1} normals)")
print(f"antithetic : {est_anti}   (n_pairs={n_pairs}, {2 * n_pairs} normals)")
print(f"SE ratio anti/plain = {est_anti.se / est_zp.se:.3f}   exact sqrt({var_ratio_exact:.4f}) = {np.sqrt(var_ratio_exact):.3f}")
print(f"Var(A): sample {A.var(ddof=1):.5f}   exact (1/4)(1-2/pi) = {varA_exact:.5f}")
var_ratio_mc = 2 * A.var(ddof=1) / np.maximum(Z, 0).var(ddof=1)
print(f"variance ratio at equal cost: sample {var_ratio_mc:.3f}   exact {var_ratio_exact:.4f}")

공짜 점심의 이유는 $Z^+$와 $(-Z)^+$의 **음의 공분산**(정리 D): 하나가 크면 다른 하나는 0이라 서로 상쇄한다. 같은 비용으로 분산이 0.533배, SE가 0.73배. C4(옵션 가격)에서 본격적으로 쓴다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
se_pi_exact = np.sqrt(theta_pi * (1 - theta_pi) / n1)          # 이 실행의 n 에서의 정확한 SE
se_pi_1e5 = np.sqrt(theta_pi * (1 - theta_pi) / 1e5)            # 예측 질문의 n = 1e5
print(f"SE of pi/4 estimate: predicted (n=1e5) {predictions['se_pi_quarter']},  "
      f"exact (n=1e5) {se_pi_1e5:.5f},  observed (n={n1}) {est_pi.se:.5f}, exact (n={n1}) {se_pi_exact:.5f}")
print(f"antithetic variance ratio: predicted {predictions['var_ratio_antithetic']},  "
      f"sample {var_ratio_mc:.3f},  exact {var_ratio_exact:.4f}")

rows = [
    {"name": "pi/4 (disk hit probability)", "predicted": None, "estimate": est_pi, "exact": theta_pi},
    {"name": "E[Z+]", "predicted": None, "estimate": est_zp, "exact": theta_zp},
    {"name": "log-log slope of RMSE vs N", "predicted": predictions["loglog_slope"],
     "estimate": slope, "se": se_slope, "exact": -0.5},
    {"name": "Wald 95% CI coverage, p=0.3, n=100", "predicted": predictions["coverage_p03_n100"],
     "estimate": cov_p03, "exact": exact_p03},
    {"name": "Wald 95% CI coverage, p=0.05, n=100", "predicted": predictions["coverage_p005_n100"],
     "estimate": cov_p005, "exact": exact_p005},
    {"name": "E[max of 10 N(0,1)]", "predicted": predictions["emax10"], "estimate": est_max10, "exact": emax10_ref},
    {"name": "P(S_k > 0 for k=1..20), N(0,1) steps", "predicted": None, "estimate": est_stay_pos, "exact": stay_pos_exact},
    {"name": "E[Z+] via antithetic pairs", "predicted": None, "estimate": est_anti, "exact": theta_zp},
]
Markdown(compare_table(rows))

각 행에서 "investigate"가 뜬다면:
- **pi/4**: `inside` 조건이 틀렸거나(부등호, 제곱) `mc_proportion`에 bool이 아닌 것을 넣었다.
- **E[Z+]**: `np.maximum(Z, 0)` 대신 `np.abs`를 썼거나 `ddof`가 어긋나 SE가 잘못됐다.
- **log-log slope**: 편향이 있거나(RMSE가 $N$에서 평평해짐), 분산이 무한하거나(E3), `polyfit`에 로그를 안 넣었다.
- **coverage p=0.3**: 구간 규칙이 `mc_proportion`과 다르다(1.96 대신 2, 1/n 하한 누락).
- **coverage p=0.05**: 위와 같음. 특히 $\hat p=0$인 반복에서 SE=0으로 처리하면 포함률이 더 내려간다.
- **E[max of 10]**: `max(axis=1)`의 축이 틀렸거나 정규 대신 다른 분포를 썼다.
- **stay positive**: `.all(axis=1)`이 `.any`로 바뀌었거나 걸음 수/축이 틀렸다.
- **E[Z+] antithetic**: 짝의 평균을 2로 안 나눴거나(값이 2배), 분산 감소 자체는 불편성과 무관하므로 값이 어긋나면 코드 오류다.

In [ ]:
assert_close(est_pi, theta_pi, k=4, name="pi/4")
assert_close(est_zp, theta_zp, k=4, name="E[Z+]")
assert_close(slope, -0.5, se=se_slope, k=4, name="log-log slope")
assert_close(cov_p03, exact_p03, k=4, name="coverage p=0.3")
assert_close(cov_p005, exact_p005, k=4, name="coverage p=0.05")
assert_close(est_max10, emax10_ref, k=4, name="E[max of 10 normals]")
assert_close(est_stay_pos, stay_pos_exact, k=4, name="stay positive 20 steps")
assert_close(est_anti, theta_zp, k=4, name="E[Z+] antithetic")
print("all assert_close(k=4) checks passed")

## 6. 연습문제

### E1 계산

(a) 베르누이 실험 $n=400$회 중 성공 100회: $\hat p$, $\widehat{\mathrm{SE}}$, 95% 구간을 손으로 구하라.
(b) 구간 폭을 절반으로 줄이려면 $n$은 몇 배가 필요한가?
(c) 어떤 $p$인지 모를 때 반폭 0.01의 95% 구간을 보장하는 최소 $n$은?
(d) $E[Z^+]$ 추정에서 SE 0.001을 얻기 위한 $n$은? ($\sigma^2=1/2-1/(2\pi)$)

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\hat p=100/400=0.25$, $\widehat{\mathrm{SE}}=\sqrt{0.25\cdot0.75/400}=\sqrt{0.75}/40=0.02165$, 구간 $0.25\pm1.96\cdot0.02165=0.25\pm0.04244=[0.2076,\,0.2924]$.

(b) 폭 $\propto\widehat{\mathrm{SE}}\propto1/\sqrt n$이므로 폭을 절반으로 줄이려면 $n$을 **4배** (1600).

(c) 최악의 분산은 $p=1/2$에서 $\sigma^2=1/4$, $\sigma=1/2$: $n=(1.96\cdot0.5/0.01)^2=98^2=9604$. (여론조사에서 "$n\approx1000$이면 오차 $\pm3\%$"가 나오는 계산이 바로 이것이다: $1.96\cdot0.5/\sqrt{1000}=0.031$.)

(d) $\mathrm{SE}=\sigma/\sqrt n=0.001\Rightarrow n=\sigma^2/10^{-6}=0.340845\times10^6\approx3.4\times10^5$.

```python
se_a = np.sqrt(0.25 * 0.75 / 400)
print(0.25, se_a, 0.25 - 1.96 * se_a, 0.25 + 1.96 * se_a)      # 0.25 0.02165 0.20756 0.29244
print((1.96 * 0.5 / 0.01) ** 2)                                  # 9604.0
print((0.5 - 1 / (2 * np.pi)) / 0.001**2)                        # 340845.06
```

</details>

### E2 유도 후 시뮬레이션 검증

정리 D를 완성하라: $\mathrm{Cov}(Z^+,(-Z)^+)=-1/(2\pi)$를 보이고 $\mathrm{Var}\bigl(\tfrac12[Z^++(-Z)^+]\bigr)=\tfrac14(1-2/\pi)$를 유도한 뒤, $n=10^5$ 짝으로 표본분산을 구해 4 SE 안에서 맞는지 확인하라. (표본분산의 SE는 $\sqrt{(\mu_4-\sigma^4)/n}$, $\mu_4$는 중심 4차 모멘트다. $|Z|$의 $\mu_4=0.51091$, $\sigma^4=0.13204$를 쓰거나 — $A=|Z|/2$이므로 4로 나눈다 — 더 간단히 `mc_estimate((A - A.mean())**2)`의 `se`를 써라.)

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**공분산.** $Z^+\cdot(-Z)^+=Z^+Z^-=0$이 항상 성립하므로 $E[Z^+(-Z)^+]=0$. $-Z\overset d=Z$이므로 $E[(-Z)^+]=E[Z^+]=\int_0^\infty z\varphi(z)\,dz=1/\sqrt{2\pi}$. 따라서 $\mathrm{Cov}=0-\dfrac1{\sqrt{2\pi}}\cdot\dfrac1{\sqrt{2\pi}}=-\dfrac1{2\pi}\approx-0.15915$.

**분산.** $E[(Z^+)^2]=\tfrac12E[Z^2]=\tfrac12$ (대칭)이므로 $\mathrm{Var}(Z^+)=\tfrac12-\tfrac1{2\pi}$. 정리 D의 식에 대입:
$$\mathrm{Var}(A)=\tfrac12\Bigl[\tfrac12-\tfrac1{2\pi}-\tfrac1{2\pi}\Bigr]=\tfrac14-\tfrac1{2\pi}=\tfrac14\Bigl(1-\tfrac2\pi\Bigr)\approx0.090845.$$
(확인: $A=|Z|/2$이므로 직접 $\mathrm{Var}(A)=\tfrac14[E Z^2-(E|Z|)^2]=\tfrac14[1-2/\pi]$. 같은 값.)

**시뮬레이션.** 표본분산의 SE: $|Z|$에 대해 $\sqrt{(0.51091-0.13204)/10^5}=0.001946$, $A=|Z|/2$의 분산은 $1/4$배이므로 SE도 $1/4$: $\approx0.00049$.

```python
A = np.abs(rng.standard_normal(100_000)) / 2
sv = mc_estimate((A - A.mean()) ** 2)        # 표본분산 (거의 ddof=0) 과 그 SE
print(sv, 0.25 * (1 - 2 / np.pi))            # 예: 0.0910 ± 0.0005 (n=100000)  vs 0.090845
assert_close(sv, 0.25 * (1 - 2 / np.pi), k=4, name="Var(A)")
```

</details>

### E3 가정을 깨보기

**독립성을 깨자.** AR(1) 열 $X_t=\rho X_{t-1}+\sqrt{1-\rho^2}\,\varepsilon_t$ ($\rho=0.9$, $X_0\sim N(0,1)$, 정상분산 1, 평균 0)의 길이 $n=1000$ 표본에 "순진한" 구간 $\bar X\pm1.96\,s/\sqrt n$을 적용해 $R=2000$번 반복 포함률을 구하라. 0.95에서 얼마나 떨어지는가? 참고값을 계산하라: $\mathrm{Var}(\bar X_n)=\dfrac1{n^2}\Bigl[n+2\sum_{k=1}^{n-1}(n-k)\rho^k\Bigr]$ (정확), 포함률 $\approx2\Phi\bigl(1.96/(\sqrt n\,\mathrm{sd}(\bar X_n))\bigr)-1$.

**(선택) 유한 분산을 깨자.** Pareto($\alpha=1.5$; `rng.pareto(1.5, size) + 1`, 평균 $\alpha/(\alpha-1)=3$, 분산 $\infty$)의 평균을 4.2절처럼 추정해 로그-로그 기울기가 $-1/2$보다 완만해짐을 보라(이론 $-(1-1/\alpha)=-1/3$; 유한 $N$의 추정치는 시드에 따라 대략 $-0.25\sim-0.55$로 크게 요동한다 — 그 요동 자체가 무한 분산의 증상이다).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**AR(1).** $s^2\approx1$이므로 순진한 SE $\approx1/\sqrt{1000}=0.0316$이지만 실제 $\mathrm{sd}(\bar X_n)=\sqrt{0.018820}=0.1372$: **4.34배** 과소평가. 순진한 구간이 $|\bar X|\le1.96\cdot0.0316$일 확률은 $\bar X\sim N(0,0.1372^2)$로 근사하면 $2\Phi(1.96/4.338)-1=0.3486$ (점근식 $\mathrm{Var}(\bar X_n)\approx\frac1n\frac{1+\rho}{1-\rho}=19/n$을 쓰면 0.3470). 참고값 0.3486은 $s^2=1$ 가정의 근사다: 실제로는 $E[s^2]\approx1-\mathrm{Var}(\bar X_n)\approx0.98$이라 순진한 구간이 조금 더 좁고, 고정밀 MC($R=10^5$)로 잰 포함률은 약 0.339다. $R=2000$이면 $0.33\sim0.36$($\pm0.011$)이 나오는 것이 정상이니 0.33이 나와도 코드를 의심하지 마세요.

```python
# helper (spkit에 없어서 여기서 정의)
def ar1_paths(R, n, rho, rng):
    """정상 AR(1) 경로 R 개 (R, n): X0 ~ N(0,1), 시간 루프만 파이썬."""
    X = np.empty((R, n))
    X[:, 0] = rng.standard_normal(R)
    c = np.sqrt(1 - rho**2)
    for t in range(1, n):
        X[:, t] = rho * X[:, t - 1] + c * rng.standard_normal(R)
    return X

R3, n3, rho = 2000, 1000, 0.9
X3 = ar1_paths(R3, n3, rho, rng)
xbar = X3.mean(axis=1)
naive_se = X3.std(axis=1, ddof=1) / np.sqrt(n3)
cov_naive = mc_proportion(np.abs(xbar - 0.0) <= 1.96 * naive_se)

k = np.arange(1, n3)
var_xbar = (n3 + 2 * ((n3 - k) * rho**k).sum()) / n3**2          # 0.018820
factor = np.sqrt(var_xbar) * np.sqrt(n3)                          # 4.338
cov_ref = 2 * norm.cdf(1.96 / factor) - 1                         # 0.3486
print(f"naive coverage {cov_naive}   reference {cov_ref:.4f}   sd(xbar) MC {xbar.std(ddof=1):.4f} exact {np.sqrt(var_xbar):.4f}")
```

교훈: 상관 표본에서는 유효 표본 수가 $n_{\text{eff}}\approx n\,\frac{1-\rho}{1+\rho}=n/19$로 준다. 01f(MCMC)의 자기상관 시간(autocorrelation time)이 정확히 이 보정이다. 해결책은 (i) 독립 경로를 여러 개 돌려 경로별 평균에 `mc_estimate`를 쓰거나, (ii) 배치 평균(batch means)으로 SE를 추정하는 것이다.

**Pareto (선택).**

```python
rm = []
for N in Ns_all:
    est = (rng.pareto(1.5, (200, int(N))) + 1).mean(axis=1)
    rm.append(np.sqrt(np.mean((est - 3.0) ** 2)))
print(np.polyfit(np.log(Ns_all), np.log(rm), 1)[0])           # 예: -0.42 (다른 시드에서 -0.26): 이론 -1/3, 요동이 크다
```

정규 걸음에서는 기울기가 $-0.50\pm0.01$로 안정적이었지만 Pareto(1.5)에서는 $-0.3$ 근처를 크게 요동한다. 무한 분산이면 CLT가 아니라 안정분포 극한($\alpha$-stable)이 적용되고, 오차는 $N^{-(1-1/\alpha)}=N^{-1/3}$로 훨씬 느리게 준다. 이런 경우 `mc_estimate`의 SE는 의미가 없다(표본분산이 $n$과 함께 계속 커진다).

</details>

## 7. 정리

1. MC 추정량은 불편이고 분산이 $\sigma^2/n$: $\widehat{\mathrm{SE}}=s/\sqrt n$ (ddof=1), 95% 구간 $=\hat\theta\pm1.96\,s/\sqrt n$ (`mc_estimate` / `mc_proportion` / `.ci95`).
2. 오차는 $1/\sqrt n$으로 준다(로그-로그 기울기 $-0.5$): 정밀도 10배 = 표본 100배. FAST 모드는 표본 1/10, SE $\sqrt{10}$배.
3. 포함률은 **반복 실험의 성질**이며 근사값이다: Wald 구간은 $p=0.3,n=100$에서 0.950, $p=0.05,n=100$에서 0.877 (이항 pmf로 열거하면 정확히 계산된다).
4. `assert_close(k=4)`는 SE에 상대적이므로 표본 크기와 무관하게 거짓 경보 확률이 같다($6\times10^{-5}$); 3 SE 밖은 버그 아니면 가정 오해(상관 표본, 무한 분산)다.
5. 닫힌 형식이 없어도 MC는 두 줄이다($E[\max\text{ of }10]\approx1.5388$, 20걸음 양수 유지 $\approx0.1254$); 안티테틱 짝은 $E[Z^+]$의 분산을 같은 비용에서 0.533배로 줄인다.

**trap 카드**
- Q: 표본을 10배 늘렸는데 오차막대가 10배 줄지 않았다. 코드가 틀렸나?
- A: 아니다. SE는 $1/\sqrt n$이므로 $\sqrt{10}\approx3.2$배만 준다. 자릿수 하나(10배 정밀)를 더 얻으려면 표본이 100배 필요하다. 그래서 분산 감소 기법(안티테틱, 제어변량)이 존재한다.

**다음 노트북: 00c** — 이제 시뮬레이션 결과를 SE 단위로 믿을 수 있게 되었다. 다음은 확률 도구 자체다: 조건부 기대를 확률변수로 다루기(첫걸음 분석과 마팅게일의 언어), 랜덤 합의 분산과 pgf(분기 과정), 지수분포의 무기억성과 경주(푸아송·CTMC의 심장). 00c가 그것을 시뮬레이션과 함께 정리한다.

**log/progress.md 한 줄 템플릿**

`| YYYY-MM-DD | 00b | __분 | _/5 | (예: 포함률 0.877이 왜 0.95가 아닌지 / 기울기 -1/2) |`